# Segmentation Benchmark — LandCover.ai on Colab A100

验证 Paper 12 的 PEFT 排序（Houlsby > LoRA > Linear Probe）在语义分割任务上是否成立。

- **数据集**: LandCover.ai（6 类土地覆盖分割，高分辨率航空影像 RGB）
- **方法**: Linear Probe / Houlsby / LoRA (r=8)
- **指标**: mIoU
- **3 methods × 1 modality (rgb) × 3 seeds = 9 runs**，预计 ~3-4 GPU-hours

## 0. 环境准备

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/paper12_results'
WEIGHTS_DIR = '/content/drive/MyDrive/paper12_weights'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results: {RESULTS_DIR}')

## 1. 克隆仓库 + 安装依赖

In [ ]:
%cd /content
!rm -rf AlphaEarth-System
!git clone https://github.com/zhouning/alphaearth-training-system.git AlphaEarth-System
%cd /content/AlphaEarth-System
!pip install -q -e '.[bench]'
!git log --oneline -3

In [ ]:
# 验证 torchgeo 安装 + LandCoverAI 可用
import torchgeo
print(f'torchgeo version: {torchgeo.__version__}')
from torchgeo.datasets import LandCoverAI
print('LandCoverAI import OK')

## 2. 准备 Prithvi 权重

In [ ]:
import shutil

CACHED = f'{WEIGHTS_DIR}/Prithvi_100M.pt'
LOCAL  = '/content/AlphaEarth-System/data/weights/prithvi/Prithvi_100M.pt'
os.makedirs(os.path.dirname(LOCAL), exist_ok=True)

if os.path.exists(CACHED):
    print(f'Using cached weights: {CACHED}')
    shutil.copy(CACHED, LOCAL)
else:
    print('Downloading from HuggingFace...')
    !pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(repo_id='ibm-nasa-geospatial/Prithvi-100M', filename='Prithvi_100M.pt')
    shutil.copy(downloaded, LOCAL)
    shutil.copy(downloaded, CACHED)

print(f'Weights: {LOCAL} ({os.path.getsize(LOCAL)/1e6:.1f} MB)')

In [ ]:
import yaml, glob
for cfg_path in glob.glob('geoadapter/bench/configs/*.yaml'):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    if 'prithvi' in cfg:
        cfg['prithvi']['checkpoint'] = LOCAL
        with open(cfg_path, 'w') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print(f'Patched: {cfg_path}')

## 3. 下载 LandCover.ai

LandCover.ai 约 2GB，放 Colab 本地 SSD。torchgeo 首次运行自动下载。

In [ ]:
LCAI_ROOT = '/content/landcoverai'
os.makedirs(LCAI_ROOT, exist_ok=True)

# 注入 dataset_root
cfg_path = 'geoadapter/bench/configs/landcoverai.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)
cfg['experiment']['dataset_root'] = LCAI_ROOT
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f'Dataset root: {LCAI_ROOT}')

total, used, free = shutil.disk_usage('/content')
print(f'Disk: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

## 4. 运行分割基准 (~3-4h)

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M')
out_json = f'{RESULTS_DIR}/landcoverai_segmentation.json'

# 删除之前的无效结果（如果有）
import os
if os.path.exists(out_json):
    os.remove(out_json)
    print(f'Removed stale results: {out_json}')

!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/landcoverai.yaml \
    --output {out_json} 2>&1 | tee {RESULTS_DIR}/landcoverai_{ts}.log

print(f'\n=> Results: {out_json}')

## 5. 结果分析

In [ ]:
import json
import pandas as pd

with open(out_json) as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
agg = df.groupby('method')['mIoU'].agg(['mean', 'std', 'count']).round(4)
agg.columns = ['mIoU_mean', 'mIoU_std', 'n_seeds']
print('\n=== LandCover.ai Segmentation Benchmark ===')
print(agg)
print(f'\nPaper 12 classification ranking: Houlsby >> BitFit > LoRA ~ Linear Probe')
print('Does the same ranking hold for segmentation?')

In [ ]:
import matplotlib.pyplot as plt

methods = agg.index.tolist()
means = agg['mIoU_mean'].tolist()
stds = agg['mIoU_std'].tolist()

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(methods, means, yerr=stds, capsize=5, color=['C4', 'C2', 'C1'])
ax.set_ylabel('mIoU')
ax.set_title('LandCover.ai Segmentation (6 classes, RGB)')
ax.set_ylim(0, 1.0)
ax.grid(True, axis='y', ls=':', alpha=0.5)
fig.tight_layout()

fig_path = f'{RESULTS_DIR}/landcoverai_segmentation.pdf'
fig.savefig(fig_path)
print(f'Figure saved: {fig_path}')
plt.show()